# Dynamic Pricing Env

> Overarching DP env

In [ ]:
#| default_exp envs.pricing.dynamic

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, Literal

from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss, quantile_loss
from ddopai.envs.pricing.base import BasePricingEnv
import gymnasium as gym

import numpy as np
import time

In [ ]:

# | export
class DynamicPricingEnv(BasePricingEnv):
    """
    Class implementing the dynamic pricing and learning problem, working for the single- and multi-item case.
    If alpha and beta are scalars and they are multiple SKUs, then the same parameters are used for all SKUs.
    If alpha and beta are arrays, then they should have the same length as the number of SKUs.
    Num_SKUs can be set as parameter or inferrred from the DataLoader.
    """
    def __init__(self,
        p_bound_low: Union[np.ndarray, Parameter, int, float] = 0.0, # lower price bound per SKUs
        p_bound_high: Union[np.ndarray, Parameter, int, float] = 1.0, # upper price bound per SKUs
        dataloader: BaseDataLoader = None, # dataloader TODO: replace with pricing dataloader
        gamma: float = 1, # discount factor
        
        #inv: Union[np.ndarray, Parameter, int, float] = 100, # inventory per SKUs
        horizon_train: int | str = "use_all_data", # if "use_all_data" then horizon is inferred from the DataLoader
        postprocessors: list[object] | None = None, # default is empty list 
        mode: str = "train", 
        return_truncation: str = False, # TODO:Why is this a string?
        train_tasks: list[dict] | None = None, # default is empty list
        val_tasks: list[dict] | None = None, # default is empty list
        test_tasks: list[dict] | None = None, # default is empty list
        ) -> None:

        self.print=False
        
        if dataloader is None:
            dataloader = self.update_dataloader()
        
        self.set_param("p_bound_low", p_bound_low, shape=(1,), new=True)
        self.set_param("p_bound_high", p_bound_high, shape=(1,), new=True)
        
        self.set_param("train_tasks", train_tasks, shape=(len(train_tasks),), new=True)
        self.set_param("val_tasks", val_tasks, shape=(len(val_tasks),), new=True)
        self.set_param("test_tasks", test_tasks, shape=(len(test_tasks),), new=True)
        
        self.set_param("episode", 0, (1,), new=True)
        self.set_param("task", self.train_tasks[0], new=True)
        
        
        self.set_param("horizon_train", horizon_train, new=True)

        low = np.min(dataloader.X, axis=0)
        high = np.max(dataloader.X, axis=0)
        feature_shape = dataloader.X_shape[1:]
        # Set the observation space without the SKU dimension.
        self.set_observation_space(feature_shape=feature_shape, feature_low=low, feature_high=high)
        self.set_action_space(dataloader.Y_shape, low=self.p_bound_low, high=self.p_bound_high)
        self.info_history = []
        
        mdp_info = MDPInfo(self.observation_space, self.action_space, gamma=gamma, horizon=horizon_train)
        self.update_episode_params()
        super().__init__(mdp_info=mdp_info,
                         postprocessors=postprocessors,
                         mode=mode, return_truncation=return_truncation,
                         dataloader=dataloader,
                         horizon_train=horizon_train)

    
    
    def set_observation_space(self, feature_shape, feature_low = -np.inf, feature_high = np.inf):
        
        '''
        Set the observation space of the environment.

        '''
        spaces = {
            "features": gym.spaces.Box(
                low=feature_low,
                high=feature_high,
                shape=feature_shape,
                dtype=np.float32
            ),
            "inventory": gym.spaces.Box(
                low=0.0,
                high=1.0,
                shape=(1,),
                dtype=np.float32
            ),
        }
        
        self.observation_space = gym.spaces.Dict(spaces)
        
        
        
    def step_(self,
              action: np.ndarray # prices)
                ) -> Tuple[np.ndarray, float, bool, bool, dict]:
        """
        Step function implementing the dynamic pricing and learning problem. Note that the dataloader will return an observation and a demand function.
        """
        if action.ndim == 2 and action.shape[0] == 1:
            action = np.squeeze(action, axis=0)


        observation, (reward_functions, alpha, beta) = self.get_observation()
        x = observation["features"]
        demand, true_demand = reward_functions(x, action)

        if self.task["inv"]:
            if (demand / self.inv) >= self.relative_inv:
                demand = self.relative_inv * self.inv
                if (true_demand / self.inv) >= self.relative_inv:
                    true_demand = self.relative_inv * self.inv
                self.relative_inv = np.zeros(self.relative_inv.shape, dtype=np.float32)
            else:
                self.relative_inv -= demand / self.inv
        
        reward = (demand * action)[0]
        true_reward = (true_demand * action)[0]
        terminated = True if self.relative_inv == 0 else False
        truncated = self.set_index()

        info = dict(
            inv=self.inv * self.relative_inv,
            demand=demand,
            true_demand=true_demand,
            action=action.copy(),
            reward=reward,
            true_reward=true_reward,
            alpha=alpha,
            beta=beta,
        )
        
        self.info_history.append(info)
        
        if truncated:
            if self.mode in ["test", "val"]:
                observation = {"features": np.zeros_like(x), "inventory": np.zeros_like([self.relative_inv])}
            else:
                observation = {"features": np.zeros_like(x), "inventory": np.zeros_like([self.relative_inv])}
            return observation, reward, terminated, truncated, info
        else:
            observation, _ = self.get_observation()
            if self.print:
                print("next_period:", self.index + 1)
                print("next observation:", observation)
                time.sleep(3)
            return observation, reward, terminated, truncated, info
    
    def get_observation(self):
        """
        Function to get the observation from the dataloader.
        """
        x, reward_functions = self.dataloader[self.index]
        current_inv = np.array([self.relative_inv], dtype=np.float32)

        observation = {
            "features": x,
            "inventory": current_inv
        }
        return observation, reward_functions
    
    def reset(self, start_index=None, state=None):
        """
        Reset environment to initial state.
        """
        super().reset(start_index=0, state=state)
        
        self.info_history = []


        observation, _ = self.get_observation()
        return observation
    
    def reset_episode(self, episode_index = None):
        super().reset_episode(episode_index)
        self.update_episode_params()
    
    def update_episode_params(self):
        """
        Update the parameters of the episode.
        """
        inv = np.array(self.task["inv_level"])
        relative_inv = np.ones_like(inv, dtype=np.float32)
        if hasattr(self, "inv"):
            self.set_param("inv", inv, inv.shape, new=False)
            self.set_param("relative_inv", relative_inv, relative_inv.shape, new=False)
        else:
            self.set_param("inv", inv, inv.shape, new=True)
            self.set_param("relative_inv", relative_inv, relative_inv.shape, new=True)
    
    def get_info(self):
        """
        Get the information of the environment.
        """
        return self.info_history